# 🏆 SportNewsAI — Classificazione Automatica di Articoli Sportivi

**Corso:** Fisica dei sistemi neurali e intelligenza artificiale  
**Modello:** LinearSVC + TF-IDF  
**Categorie:** Calcio · Tennis · Pallacanestro · Rugby · Formula 1 · Baseball · Golf  
**Accuracy:** 99.13%

---

### Pipeline
```
Dataset CSV → Preprocessing → TF-IDF → LinearSVC → Valutazione → Predizione
```

## 1. Setup librerie

In [1]:
!pip install -q nltk scikit-learn pandas matplotlib seaborn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

STOP_EN = set(stopwords.words('english'))

SPORT_IT = {
    'soccer':     'Calcio',
    'tennis':     'Tennis',
    'basketball': 'Pallacanestro',
    'rugby':      'Rugby',
    'F1':         'Formula 1',
    'baseball':   'Baseball',
    'golf':       'Golf',
}

print('Librerie caricate con successo.')

ModuleNotFoundError: No module named 'seaborn'

## 2. Caricamento del dataset

Carica il file `dataset_sport_cleaned.csv` (output di `preprocess_data.py`).  
Puoi anche caricare `dataset_sport_master.csv` e verrà preprocessato automaticamente nella sezione successiva.

In [ ]:
from google.colab import files

print('Carica il file CSV del dataset...')
uploaded = files.upload()

In [ ]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f'File caricato: {filename}')
print(f'Righe: {len(df)} | Colonne: {list(df.columns)}')
df.head(3)

## 3. Analisi esplorativa del dataset (EDA)

In [ ]:
# Distribuzione delle categorie
counts = df['label'].value_counts()
labels_it = [SPORT_IT.get(l, l) for l in counts.index]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels_it, counts.values,
               color='#f97316', edgecolor='#ea6c0a', height=0.6)
ax.bar_label(bars, padding=4, fontsize=10)
ax.set_xlabel('Numero di articoli')
ax.set_title('Distribuzione articoli per categoria', fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'\nTotale articoli: {len(df)}')
print(counts.to_string())

In [ ]:
# Distribuzione lunghezza testo
text_col = 'cleaned_text' if 'cleaned_text' in df.columns else 'full_text'
df['n_words'] = df[text_col].fillna('').apply(lambda x: len(x.split()))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['n_words'], bins=40, color='#f97316', edgecolor='white', linewidth=0.5)
ax.axvline(df['n_words'].median(), color='#1a1a1a', linestyle='--',
           linewidth=1.5, label=f'Mediana: {df["n_words"].median():.0f} parole')
ax.set_xlabel('Numero di parole')
ax.set_ylabel('Frequenza')
ax.set_title('Distribuzione lunghezza articoli', fontsize=13, fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Preprocessing del testo

Il preprocessing si compone di tre passi:
1. Conversione in minuscolo
2. Rimozione di caratteri non alfabetici
3. Eliminazione delle stopwords inglesi (corpus NLTK)

Il testo sorgente è costruito come `title + full_text` (dove `full_text = title + description + content`), in modo da dare maggiore peso al titolo nella rappresentazione TF-IDF.

In [ ]:
def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [w for w in text.split() if w not in STOP_EN and len(w) > 1]
    return ' '.join(tokens)


# Se il dataset è già preprocessato usa cleaned_text, altrimenti preprocessa dal raw
if 'cleaned_text' in df.columns and df['cleaned_text'].notna().sum() > 0:
    df['cleaned_text'] = df['cleaned_text'].fillna('')
    print('Colonna cleaned_text già presente — preprocessing saltato.')
else:
    # title viene concatenato due volte per dare maggiore peso al titolo nel TF-IDF
    full_text = (
        df['title'].fillna('') + ' ' +
        df['description'].fillna('') + ' ' +
        df['content'].fillna('')
    )
    source = df['title'].fillna('') + ' ' + full_text
    print('Preprocessing in corso...')
    df['cleaned_text'] = source.apply(preprocess)
    print('Preprocessing completato.')

# Rimuovi righe vuote
df = df[df['cleaned_text'].str.strip() != ''].reset_index(drop=True)

# Esempio
print(f'\nDopo preprocessing:\n  {df["cleaned_text"].iloc[0][:120]}...')

## 5. Vettorizzazione TF-IDF

**TF-IDF** (Term Frequency – Inverse Document Frequency) rappresenta ogni articolo come vettore numerico dove ogni dimensione è una parola del vocabolario.

$$\text{TF-IDF}(t, d) = \text{TF}(t,d) \times \log\frac{N}{\text{df}(t)}$$

Con `sublinear_tf=True` si usa $\text{TF} = 1 + \log(\text{count})$ per ridurre il peso dei termini molto frequenti — particolarmente utile con articoli di lunghezza variabile come quelli da GNews (content troncato a ~253 caratteri).

In [ ]:
X_raw = df['cleaned_text']
y     = df['label']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000, sublinear_tf=True)
X_train = vectorizer.fit_transform(X_train_raw)
X_test  = vectorizer.transform(X_test_raw)

print(f'Articoli training: {X_train.shape[0]}')
print(f'Articoli test:     {X_test.shape[0]}')
print(f'Dimensione vocabolario: {X_train.shape[1]} feature')

## 6. Addestramento del modello — LinearSVC

**LinearSVC** (Support Vector Classifier lineare) cerca l'iperpiano di separazione con margine massimo tra le classi. In uno spazio TF-IDF ad alta dimensionalità, i dati sono quasi sempre linearmente separabili, rendendo il kernel lineare la scelta ottimale.

La classificazione multi-classe avviene con strategia **one-vs-rest**: per ciascuna delle 7 categorie si addestra un classificatore binario «questa categoria vs. tutte le altre».

In [ ]:
model = LinearSVC(random_state=42, max_iter=2000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print(f'>>> ACCURATEZZA GLOBALE: {acc*100:.2f}% <<<')
print(f'    Errori: {(y_test != y_pred).sum()} su {len(y_test)} articoli di test')

## 7. Valutazione dettagliata

In [ ]:
# Classification report
report = classification_report(y_test, y_pred, target_names=model.classes_)
print('Classification Report\n')
print(report)

In [ ]:
# Confusion matrix
labels_order = model.classes_
labels_it_order = [SPORT_IT.get(l, l) for l in labels_order]

cm = confusion_matrix(y_test, y_pred, labels=labels_order)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Oranges',
    xticklabels=labels_it_order,
    yticklabels=labels_it_order,
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predetto', fontsize=11)
ax.set_ylabel('Reale', fontsize=11)
ax.set_title('Matrice di Confusione', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Precision / Recall / F1 per categoria — grafico a barre
from sklearn.metrics import classification_report
rep_dict = classification_report(y_test, y_pred, output_dict=True)

classes  = model.classes_
metrics  = ['precision', 'recall', 'f1-score']
x        = np.arange(len(classes))
width    = 0.25
colors   = ['#f97316', '#fbbf24', '#1a1a1a']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = [rep_dict[c][metric] for c in classes]
    ax.bar(x + i*width, vals, width, label=metric.capitalize(),
           color=color, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels([SPORT_IT.get(c, c) for c in classes], rotation=20, ha='right')
ax.set_ylim(0.85, 1.01)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title('Precision · Recall · F1-score per categoria', fontsize=13, fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Analisi degli errori

Identifichiamo gli articoli classificati in modo errato per capire dove il modello fatica.

In [ ]:
test_df = X_test_raw.reset_index(drop=True).to_frame(name='text')
test_df['true']      = y_test.reset_index(drop=True)
test_df['predicted'] = y_pred

errors = test_df[test_df['true'] != test_df['predicted']].copy()
print(f'Errori totali: {len(errors)}\n')

# Coppie di confusione più frequenti
confusion_pairs = (
    errors.groupby(['true', 'predicted'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
confusion_pairs['true_it']      = confusion_pairs['true'].map(SPORT_IT)
confusion_pairs['predicted_it'] = confusion_pairs['predicted'].map(SPORT_IT)
print('Coppie di confusione più frequenti:')
print(confusion_pairs[['true_it','predicted_it','count']].to_string(index=False))

In [ ]:
# Mostra alcuni articoli errati
print('Esempi di articoli classificati erroneamente:\n')
for _, row in errors.head(5).iterrows():
    true_it = SPORT_IT.get(row['true'], row['true'])
    pred_it = SPORT_IT.get(row['predicted'], row['predicted'])
    print(f'  Reale: {true_it:<15} Predetto: {pred_it}')
    print(f'  Testo: {row["text"][:120]}...')
    print()

## 9. Feature importance — parole più discriminanti per categoria

LinearSVC assegna un peso a ciascuna feature (parola) per ogni classe. Le parole con peso maggiore sono le più discriminanti.

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
n_top         = 12
classes       = model.classes_

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, cls in enumerate(classes):
    coefs    = model.coef_[i]
    top_idx  = np.argsort(coefs)[-n_top:][::-1]
    top_words = feature_names[top_idx]
    top_vals  = coefs[top_idx]

    ax = axes[i]
    ax.barh(top_words[::-1], top_vals[::-1], color='#f97316', edgecolor='white')
    ax.set_title(SPORT_IT.get(cls, cls), fontweight='bold', fontsize=11)
    ax.spines[['top','right','bottom']].set_visible(False)
    ax.tick_params(axis='y', labelsize=8)
    ax.set_xticks([])

# Nasconde l'ultimo subplot vuoto
axes[-1].set_visible(False)

fig.suptitle(f'Top {n_top} parole discriminanti per categoria (pesi LinearSVC)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 10. Predizione interattiva

Inserisci un testo di articolo sportivo in inglese per ottenere la categoria predetta.

In [ ]:
def predict(text: str) -> None:
    cleaned = preprocess(text)
    if not cleaned.strip():
        print('Testo troppo breve o fuori vocabolario.')
        return

    vec    = vectorizer.transform([cleaned])
    scores = model.decision_function(vec)[0]

    # Softmax sugli score per ottenere pseudo-probabilità
    shifted = scores - scores.max()
    exps    = np.exp(shifted)
    probs   = exps / exps.sum()

    ranked = sorted(zip(model.classes_, probs), key=lambda x: x[1], reverse=True)
    best_cls, best_prob = ranked[0]

    print(f'Categoria predetta: {SPORT_IT.get(best_cls, best_cls)} ({best_prob*100:.1f}%)')
    print()
    print('Distribuzione completa:')
    for cls, prob in ranked:
        bar = '█' * int(prob * 40)
        print(f'  {SPORT_IT.get(cls, cls):<15} {bar:<40} {prob*100:5.1f}%')


# ── Test ──────────────────────────────────────────────────────────────────────
article = """
The reigning champion crashed out in the third lap after a mechanical failure
forced him to retire from the race. The safety car was deployed immediately
as marshals cleared the track.
"""

predict(article)

In [ ]:
# ── Predizione su testo personalizzato ────────────────────────────────────────
testo = input('Inserisci il testo dell\'articolo (in inglese): ')
predict(testo)